# MoA: full pipeline (CNN + CDLM + XGBoost ensemble)

## Setup

In [ ]:
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ExponentialLR
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
from sklearn.decomposition import PCA
import xgboost as xgb

DATA_DIR = "/kaggle/input/competitions/lish-moa"
SEED = 42
EPS = 1e-7
W_CNN, W_XGB, W_CDLM = 0.55, 0.25, 0.20   # found by OOF grid search

CNN_EPOCHS = 11    # average best epoch from CNN cross-validation
CDLM_EPOCHS = 14   # average best epoch from CDLM cross-validation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Load data

In [ ]:
# At submission time Kaggle swaps test_features.csv and sample_submission.csv for the full hidden
# test set, so everything below must be computed from these files, never from precomputed CSVs.
train_features = pd.read_csv(f"{DATA_DIR}/train_features.csv")
train_targets = pd.read_csv(f"{DATA_DIR}/train_targets_scored.csv")
test_features = pd.read_csv(f"{DATA_DIR}/test_features.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

target_cols = list(sample_submission.columns[1:])
assert (train_features.sig_id.values == train_targets.sig_id.values).all()

# control rows have no drug and no active targets: drop from training, predict 0 at test time
train_mask = (train_features.cp_type == "trt_cp").values
train = train_features[train_mask].reset_index(drop=True)
Y = train_targets.loc[train_mask, target_cols].values.astype(np.float32)

test_mask = (test_features.cp_type == "trt_cp").values
test = test_features[test_mask].reset_index(drop=True)
print("train rows:", len(train), "| test rows to predict:", len(test),
      "| test control rows:", (~test_mask).sum())

## Feature engineering

In [ ]:
def make_features(trn_df, tst_df, n_gene_pca=50, n_cell_pca=15, seed=SEED):
    """Quantile-transformed raw features + a few PCA components + one-hot cp_time/cp_dose.
    Everything is fit on the training rows only, then applied to the test rows."""
    g_cols = [c for c in trn_df.columns if c.startswith("g-")]
    c_cols = [c for c in trn_df.columns if c.startswith("c-")]

    qt_g = QuantileTransformer(output_distribution="normal", random_state=seed)
    qt_c = QuantileTransformer(output_distribution="normal", random_state=seed)
    g_tr, g_te = qt_g.fit_transform(trn_df[g_cols]), qt_g.transform(tst_df[g_cols])
    c_tr, c_te = qt_c.fit_transform(trn_df[c_cols]), qt_c.transform(tst_df[c_cols])

    pca_g = PCA(n_components=n_gene_pca, random_state=seed).fit(g_tr)
    pca_c = PCA(n_components=n_cell_pca, random_state=seed).fit(c_tr)

    ohe = OneHotEncoder(handle_unknown="ignore")
    cat_tr = ohe.fit_transform(trn_df[["cp_time", "cp_dose"]]).toarray()
    cat_te = ohe.transform(tst_df[["cp_time", "cp_dose"]]).toarray()

    X_tr = np.hstack([g_tr, c_tr, pca_g.transform(g_tr), pca_c.transform(c_tr), cat_tr]).astype(np.float32)
    X_te = np.hstack([g_te, c_te, pca_g.transform(g_te), pca_c.transform(c_te), cat_te]).astype(np.float32)
    return X_tr, X_te


X_tr, X_te = make_features(train, test)
print("features:", X_tr.shape, X_te.shape)

## Custom optimizer

In [ ]:
class CustomAdam(torch.optim.Optimizer):
    def __init__(self, params, lr=0.001, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.):
        assert lr > 0., "learning rate must be positive"
        assert 0. <= betas[0] < 1., "beta1 must be inside [0, 1)"
        assert 0. <= betas[1] < 1., "beta2 must be inside [0, 1)"
        assert eps > 0., "epsilon must be greater than 0"
        assert weight_decay >= 0., "weight_decay must be greater or equal to 0"
        defaults = dict(lr=lr, beta1=betas[0], beta2=betas[1], eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)

                if group['weight_decay'] != 0:
                    p.mul_(1 - group['lr'] * group['weight_decay'])

                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                t = state['step']

                exp_avg.mul_(group['beta1']).add_(p.grad, alpha=1 - group['beta1'])
                exp_avg_sq.mul_(group['beta2']).addcmul_(p.grad, p.grad, value=1 - group['beta2'])

                m_hat = exp_avg / (1 - group['beta1'] ** t)
                v_hat = exp_avg_sq / (1 - group['beta2'] ** t)
                p.sub_(group['lr'] * m_hat / (v_hat.sqrt() + group['eps']))

## Models

In [ ]:
class CNN(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, channels: int = 256, length: int = 16):
        super().__init__()
        self.channels = channels
        self.length = length

        self.norm1 = nn.BatchNorm1d(input_dim)
        self.dropout1 = nn.Dropout(0.2)
        self.linear1 = nn.Linear(input_dim, channels * length)
        self.dropout2 = nn.Dropout(0.2)

        self.norm2 = nn.BatchNorm1d(channels)
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.norm3 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        self.output = nn.Linear(channels * (length // 4), output_dim)

    def forward(self, x):
        x = self.dropout2(F.relu(self.linear1(self.dropout1(self.norm1(x)))))
        x = x.view(-1, self.channels, self.length)
        skip = x
        x = F.relu(self.conv1(self.norm2(x)))
        x = x + skip
        x = self.pool1(x)
        skip = x
        x = F.relu(self.conv2(self.norm3(x)))
        x = x + skip
        x = self.pool2(x).flatten(1)
        return self.output(x)


import torch
import torch.nn as nn
import torch.nn.functional as F


class ChannelDropout(nn.Module):
    def __init__(self, prob=0.1):
        super().__init__()
        self.prob = prob

    def forward(self, x):
        if not self.training:
            return x
        col_number = int(self.prob * x.size(1))
        col_indexes = torch.randperm(x.size(1))[:col_number]
        x = x.clone()
        x[:, col_indexes] = 0
        actual_prob = col_number / x.size(1)
        x = x / (1 - actual_prob)
        return x


class CustomLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        weight = torch.empty(self.in_features, self.out_features)
        nn.init.kaiming_uniform_(weight)
        self.weight = nn.Parameter(weight)
        bias = torch.empty(1, self.out_features)
        nn.init.uniform_(bias, 0, 0.01)
        self.bias = nn.Parameter(bias)

    def forward(self, x):
        return torch.add(torch.matmul(x, self.weight), self.bias)


class CDLM(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super(CDLM, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim

        self.norm1 = nn.BatchNorm1d(input_dim)
        self.dropout1 = ChannelDropout(0.5)
        self.linear1 = CustomLinear(input_dim, 1024)
        self.dropout2 = ChannelDropout(0.25)

        self.norm2 = nn.BatchNorm1d(1024)
        self.linear2 = CustomLinear(1024, 512)
        self.dropout3 = ChannelDropout(0.15)

        self.norm3 = nn.BatchNorm1d(512)
        self.linear3 = CustomLinear(512, 256)
        self.dropout4 = ChannelDropout(0.1)

        self.norm4 = nn.BatchNorm1d(256)
        self.linear4 = CustomLinear(256, 256)
        self.dropout5 = ChannelDropout(0.1)

        self.output = CustomLinear(256, output_dim)

    def forward(self, x):
        x = self.dropout2(F.relu(self.linear1(self.dropout1(self.norm1(x)))))

        x = self.dropout3(F.relu(self.linear2(self.norm2(x))))

        x = self.dropout4(F.relu(self.linear3(self.norm3(x))))

        x = self.dropout5(F.relu(self.linear4(self.norm4(x))))

        output = self.output(x)
        return output


## Training and prediction helpers

In [ ]:
def train_full(model, optimizer, scheduler, X, y, epochs, batch_size=256):
    """Trains on all rows for a fixed number of epochs (no validation set exists here)."""
    criterion = nn.BCEWithLogitsLoss()
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True, drop_last=True)
    for epoch in range(epochs):
        model.train()
        train_loss = 0.
        for x_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        if scheduler is not None:
            scheduler.step()
        print(f"  epoch {epoch:3d}  train {train_loss / len(loader):.5f}")
    return model


@torch.no_grad()
def predict(model, X):
    model.eval()
    return torch.sigmoid(model(X)).cpu().numpy()


X_tr_t = torch.from_numpy(X_tr).to(device)
y_tr_t = torch.from_numpy(Y).to(device)
X_te_t = torch.from_numpy(X_te).to(device)

## CNN

In [ ]:
set_seed()
cnn = CNN(input_dim=X_tr.shape[1], output_dim=len(target_cols)).to(device)
opt = CustomAdam(cnn.parameters(), lr=2e-4, weight_decay=1e-2)
print("Training CNN")
train_full(cnn, opt, ExponentialLR(opt, gamma=0.95), X_tr_t, y_tr_t, CNN_EPOCHS)
pred_cnn = predict(cnn, X_te_t)

## CDLM

In [ ]:
set_seed()
cdlm = CDLM(input_dim=X_tr.shape[1], output_dim=len(target_cols)).to(device)
opt = torch.optim.Adam(cdlm.parameters(), lr=1e-3)   # match whatever your friend's final run used
print("Training CDLM")
train_full(cdlm, opt, None, X_tr_t, y_tr_t, CDLM_EPOCHS)
pred_cdlm = predict(cdlm, X_te_t)

## XGBoost

In [ ]:
XGB_PARAMS = dict(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    tree_method="hist", device="cuda" if torch.cuda.is_available() else "cpu",
    eval_metric="logloss", verbosity=0, random_state=SEED,
)

pred_xgb = np.zeros((len(X_te), len(target_cols)), dtype=np.float64)
print("Training XGBoost")
for j, tgt in enumerate(target_cols):
    y_col = Y[:, j]
    if y_col.min() == y_col.max():
        pred_xgb[:, j] = y_col.mean()
        continue
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(X_tr, y_col)
    pred_xgb[:, j] = model.predict_proba(X_te)[:, 1]
    if (j + 1) % 50 == 0 or j == len(target_cols) - 1:
        print(f"  [{j + 1}/{len(target_cols)}] {tgt}")

## Blend and write the submission

In [ ]:
blend = W_CNN * pred_cnn + W_XGB * pred_xgb + W_CDLM * pred_cdlm
blend = np.clip(blend, EPS, 1 - EPS)

sub_trt = pd.DataFrame(blend, columns=target_cols)
sub_trt.insert(0, "sig_id", test.sig_id.values)

# every test row gets a row here: treated rows get the blend, control rows get 0
submission = sample_submission[["sig_id"]].merge(sub_trt, on="sig_id", how="left").fillna(0.0)

ctl_ids = set(test_features.loc[~test_mask, "sig_id"])
filled = submission.loc[~submission.sig_id.isin(sub_trt.sig_id), "sig_id"]
assert set(filled) <= ctl_ids, "a treated row was not predicted"
assert submission.shape == sample_submission.shape
assert list(submission.columns) == list(sample_submission.columns)

submission.to_csv("submission.csv", index=False)
print(submission.shape)
submission.head()